# Reducir a 20 variables rezagadas con criterio Spearman  


# Características clave del Script:

1. **Control de Multicolinealidad Estricto:** Evalúa secuencialmente la correlación de Spearman absoluta con la variable objetivo. Si una variable candidata tiene una correlación interna mayor a `0.70` con alguna ya seleccionada, se descarta para garantizar la máxima independencia estadística posible.
2. **Generación del Reporte en `3_resultados`:** Guarda una lista limpia en formato Excel con los coeficientes Spearman originales (con su dirección/signo) y absolutos.
3. **Generación del Dataset en `2_procesados`:** Guarda el archivo Excel final manteniendo el orden cronológico original e incluyendo exactamente `año`, `semana_epi`, `casos_dengue` y las 20 variables independientes seleccionadas.



In [9]:
import pandas as pd
import numpy as np
import os

# =============================================================================
# 1. CONFIGURACIÓN DE RUTAS DE ENTRADA Y SALIDA
# =============================================================================
ruta_entrada = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\2_datos\1_raw\2_meteo_epi_rezagos_meteo_epi.xlsx"
carpeta_procesados = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\2_datos\2_procesados"
carpeta_resultados = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\3_resultados"

# Asegurar de que las carpetas de destino existan físicamente
os.makedirs(carpeta_procesados, exist_ok=True)
os.makedirs(carpeta_resultados, exist_ok=True)

ruta_dataset_final = os.path.join(carpeta_procesados, "1_meteo_epi_spearman_meteo_epi_20.xlsx")
ruta_reporte_variables = os.path.join(carpeta_resultados, "1_lista_variables_seleccionadas_spearman_20.xlsx")

print("Cargando el archivo de datos original...")
df = pd.read_excel(ruta_entrada)

# Asegurar el preprocesamiento de la fecha y ordenamiento de la serie de tiempo
df['fecha'] = pd.to_datetime(df['fecha'])
df.set_index('fecha', inplace=True)
df = df.sort_index()

# =============================================================================
# 2. METODOLOGÍA DE FILTRADO POR CORRELACIÓN DE SPEARMAN
# =============================================================================
target = 'casos_dengue'
columnas_obligatorias = ['año', 'semana_epi']

# Extraer únicamente los predictores rezagados y contemporáneos candidatos
candidatas = [col for col in df.columns if col not in [target] + columnas_obligatorias]

print("Calculando matriz de correlación de Spearman general...")
matriz_corr = df[[target] + candidatas].corr(method='spearman')

# Ordenar las variables candidatas de Mayor a Menor según su correlación absoluta con el objetivo
corr_con_objetivo = matriz_corr[target].drop(target).abs().sort_values(ascending=False)

variables_seleccionadas = []
umbral_multicolinealidad = 0.70  # Umbral máximo permitido entre variables predictoras

print("Ejecutando algoritmo de selección independiente...")
for var in corr_con_objetivo.index:
    if len(variables_seleccionadas) >= 20:  # Detenerse al alcanzar exactamente 20 variables
        break
        
    # Verificar correlación cruzada con las variables que ya fueron aceptadas
    es_redundante = False
    for var_sel in variables_seleccionadas:
        if abs(matriz_corr.loc[var, var_sel]) > umbral_multicolinealidad:
            es_redundante = True
            break
            
    if not es_redundante:
        variables_seleccionadas.append(var)

# =============================================================================
# 3. EXPORTAR REPORTE DE VARIABLES SELECCIONADAS (A: 3_resultados)
# =============================================================================
tabla_reporte = pd.DataFrame({
    'Variable': variables_seleccionadas,
    'Correlacion_Spearman': [matriz_corr.loc[v, target] for v in variables_seleccionadas],
    'Correlacion_Absoluta': [corr_con_objetivo[v] for v in variables_seleccionadas]
})

print(f"Guardando reporte de variables seleccionadas en:\n-> {ruta_reporte_variables}")
tabla_reporte.to_excel(ruta_reporte_variables, index=False)

# Consola de control rápida para inspección metodológica
print("\n" + "="*65)
print(f"RESUMEN: 20 VARIABLES SELECCIONADAS (MÁXIMA INDEPENDENCIA)")
print("="*65)
for idx, fila in tabla_reporte.iterrows():
    print(f"{idx+1:02d}. {fila['Variable']:<32} | Coef. Spearman: {fila['Correlacion_Spearman']:.4f}")
print("="*65)

# =============================================================================
# 4. CONSTRUCCIÓN Y EXPORTACIÓN DEL DATASET REDUCIDO (A: 2_procesados)
# =============================================================================
# Combinar identificadores obligatorios, objetivo y las 20 dimensiones filtradas
columnas_finales = columnas_obligatorias + [target] + variables_seleccionadas
df_final = df[columnas_finales].copy()

print(f"\nGuardando dataset final reducido de 20 variables en:\n-> {ruta_dataset_final}")
# Reajustar índice para conservar el campo 'fecha' como columna estándar en el archivo final de Excel
df_final.reset_index().to_excel(ruta_dataset_final, index=False)

print("\n=== PROCESO COMPLETADO EXITOSAMENTE ===")
print(f"Dataset guardado con dimensiones: {df_final.shape[0]} filas por {df_final.shape[1] + 1} columnas (incluyendo fecha).")


Cargando el archivo de datos original...
Calculando matriz de correlación de Spearman general...
Ejecutando algoritmo de selección independiente...
Guardando reporte de variables seleccionadas en:
-> C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_entrenamiento_modelo_sat\3_resultados\1_lista_variables_seleccionadas_spearman_20.xlsx

RESUMEN: 20 VARIABLES SELECCIONADAS (MÁXIMA INDEPENDENCIA)
01. casos_dengue_lag_1               | Coef. Spearman: 0.9245
02. hum_esp_lag_9                    | Coef. Spearman: 0.5529
03. hum_esp_lag_3                    | Coef. Spearman: 0.5398
04. hum_rel_lag_11                   | Coef. Spearman: 0.4012
05. vel_vi_max_lag_4                 | Coef. Spearman: 0.3501
06. dias_lluvia_lag_10               | Coef. Spearman: 0.3392
07. dias_lluvia_lag_9                | Coef. Spearman: 0.3372
08. dias_lluvia_lag_7                | Coef. Spearman: 0.3327
09. vel_vi_max_lag_7                 | Coef. Spearman: 0.3314
10. dia